In [34]:
import random

from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

import graphviz
from tqdm import tqdm
from sklearn.metrics import f1_score

from matplotlib import pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.tree import export_graphviz

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.datasets import make_circles
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import mean_absolute_percentage_error as MAPE
import catboost

In [35]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [36]:
train_df

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125


In [37]:
test_df

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,2915,160,RM,21.0,1936,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2006,WD,Normal
1455,2916,160,RM,21.0,1894,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2006,WD,Abnorml
1456,2917,20,RL,160.0,20000,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,9,2006,WD,Abnorml
1457,2918,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal


необходимо перевести все данные в числовые значения (где они не числовые изначально)

In [38]:
categorical_features_train = train_df.columns.tolist()
categorical_features_test =	test_df.columns.tolist()
encoders = {}

for feature in categorical_features_train:
    encoder = LabelEncoder()
    train_df[feature] = encoder.fit_transform(train_df[feature])
    
    encoders[feature] = encoder

for feature in categorical_features_test:
    encoder = LabelEncoder()
    test_df[feature] = encoder.fit_transform(test_df[feature])
    
    encoders[feature] = encoder

In [39]:
y_df = train_df['SalePrice']
x_df = train_df.drop('SalePrice', axis=1)

In [40]:
x, x_valid, y, y_valid = train_test_split(x_df, y_df, test_size=0.2, random_state=42)

In [41]:
import optuna
from catboost import CatBoostClassifier
from catboost import CatBoostRegressor
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


def objective(trial):
    # Определение гиперпараметров, которые будут оптимизированы
    params = {
        'iterations': trial.suggest_int('iterations', 50, 500),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-5, 1e2, log=True),
        'border_count': trial.suggest_int('border_count', 1, 255),
        'thread_count': -1,
        'loss_function': 'MultiClass',
        'eval_metric': 'MultiClass',
        'verbose': False,
    }

    # Создание и обучение модели CatBoost
    model = CatBoostClassifier(**params, random_state=42)
    model.fit(x, y, eval_set=(x_valid, y_valid), early_stopping_rounds=20, verbose_eval=False)

    # Предсказание на тестовом наборе данных
    y_pred = model.predict(x_valid)

    # Оценка качества модели
    accuracy = accuracy_score(y_valid, y_pred)

    return 1.0 - accuracy  # Optuna минимизирует целевую функцию, поэтому используем (1 - accuracy)

# Создание и запуск оптимизации с помощью Optuna
study = optuna.create_study(direction='minimize')  # 'minimize' для задачи минимизации ошибки
study.optimize(objective, n_trials=100)

# Получение лучших гиперпараметров
best_params = study.best_params
print(f'Best Hyperparameters: {best_params}')

# Получение лучшей ошибки (1 - accuracy)
best_error = study.best_value
print(f'Best Error: {best_error}')


[I 2024-01-27 15:24:44,829] A new study created in memory with name: no-name-748bffa4-7b14-4628-8711-22691198ecd0


[W 2024-01-27 15:24:45,208] Trial 0 failed with parameters: {'iterations': 269, 'depth': 4, 'learning_rate': 0.04272839773814448, 'l2_leaf_reg': 0.06816006164045159, 'border_count': 202} because of the following error: CatBoostError('C:/Go_Agent/pipelines/BuildMaster/catboost.git/catboost/private/libs/algo/data.cpp:196: Dataset test #0 contains class label "246" that is not present in the learn dataset').
Traceback (most recent call last):
  File "c:\Users\korde\anaconda3\Lib\site-packages\optuna\study\_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\korde\AppData\Local\Temp\ipykernel_7112\3737274245.py", line 25, in objective
    model.fit(x, y, eval_set=(x_valid, y_valid), early_stopping_rounds=20, verbose_eval=False)
  File "c:\Users\korde\anaconda3\Lib\site-packages\catboost\core.py", line 5100, in fit
    self._fit(X, y, cat_features, text_features, embedding_features, None, sample_weight, None, None, None,

CatBoostError: C:/Go_Agent/pipelines/BuildMaster/catboost.git/catboost/private/libs/algo/data.cpp:196: Dataset test #0 contains class label "246" that is not present in the learn dataset

In [46]:
params = {
        'iterations': 500,
        'depth': 10,
        'learning_rate': 0.05,
        'l2_leaf_reg': 1e-5,
        'border_count': 1,
        'thread_count': -1,
        'loss_function': 'MultiRMSE',
        'eval_metric': 'MultiRMSE',
        'verbose': False,
    }

cat_regres = CatBoostRegressor(**params, random_state=42)
cat_class = CatBoostClassifier(**params, random_state=42)
cat_regres.fit(x, y)
# cat_class.fit(x, y) # Error: Target with classes must contain only 2 unique values for binary classification
y_pred = abs(cat_regres.predict(y_valid))

In [48]:
y_valid

892     246
1105    592
413     106
522     260
1036    582
       ... 
479      46
1361    513
802     366
651      89
722     135
Name: SalePrice, Length: 292, dtype: int64

submission

In [ ]:
submission = pd.read_csv('sample_submission.csv')

# assert len(test_df) == len(submission)
# assert np.all(submission['Id'] == test_df.index)

# change train_df['SalePrice'].min() to prediction of cool model 
submission['SalePrice'] = train_df['SalePrice'].min()
submission.to_csv("submission.csv", index=False)